In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/input-filess/Emotion_Vocab.txt
/kaggle/input/input-filess/training_data_sarcasam.txt
/kaggle/input/input-filess/testing_data_sarcasam.txt
/kaggle/input/input-filess/emotion_spelling_final.txt


In [2]:

#GENERAL CODE FOR TEAM_MATES TO USE AND NORMALISE ANY SENTENCE

#CSV FILE MUST BE LOADED FIRST 
#IN KAGGLE/INPUT AND YOU MUST GIVE PATH  TO FUNCTION
#AND CALL THE FUNCTION THAT WILL NORMALISE THE SENTENCE

import pandas as pd
def load_emotion_dictionaries(csv_file_path):
    
    # Loads the emotion dictionary from CSV and creates spelling normalization dictionaries
    
    # Parameters:
    # csv_file_path (str): Path to the emotion dictionary CSV file
    
    # Returns:
    # dict: Combined dictionary for text normalization
    
    
    # Read the CSV file
    df = pd.read_csv(csv_file_path)
    
    # Create emotion dictionaries
    emotion_dicts = {
        'joy': {}, 'sadness': {}, 'anger': {}, 'sarcasm': {}, 'neutral': {}
    }
    
    # Build the dictionaries
    for index, row in df.iterrows():
        emotion = row['emotion_category']
        correct = row['correct_spelling']
        
        # Add all spelling variations
        emotion_dicts[emotion][row['wrong_spelling1']] = correct
        emotion_dicts[emotion][row['wrong_spelling2']] = correct
        emotion_dicts[emotion][row['wrong_spelling3']] = correct
        emotion_dicts[emotion][row['wrong_spelling4']] = correct
        emotion_dicts[emotion][row['wrong_spelling5']] = correct
        emotion_dicts[emotion][correct] = correct
    
    # Create one combined dictionary for easy use
    combined_dict = {}
    for emotion_dict in emotion_dicts.values():
        combined_dict.update(emotion_dict)
    
    return combined_dict


def normalize_text(text, spelling_dict):
    
    # Normalizes Roman Urdu text by correcting spelling variations
    
    # Parameters:
    # text (str): Input text to normalize
    # spelling_dict (dict): Dictionary with spelling corrections
    
    # Returns:
    # str: Normalized text with corrected spellings
    
    
    words = text.split()
    normalized_words = []
    
    for word in words:
        # Convert to lowercase and check in dictionary
        normalized_word = spelling_dict.get(word.lower(), word)
        normalized_words.append(normalized_word)
    
    return ' '.join(normalized_words)



        
      



In [3]:
import pandas as pd

# 1️⃣ Load emotion dictionaries from CSV with error handling
def load_emotion_dictionaries(csv_file_path):
    try:
        # Read CSV with error_bad_lines parameter for handling inconsistent rows
        df = pd.read_csv(csv_file_path, on_bad_lines='skip')
        
        emotion_dicts = {
            'joy': {}, 'sadness': {}, 'anger': {}, 'sarcasm': {}, 'neutral': {}
        }
        
        for _, row in df.iterrows():
            emotion = row.get('emotion_category', 'neutral')
            correct = row.get('correct_spelling', '')
            
            if pd.isna(emotion) or pd.isna(correct):
                continue
                
            # Add all wrong spellings
            for i in range(1, 6):
                col_name = f'wrong_spelling{i}'
                if col_name in df.columns and pd.notna(row[col_name]):
                    emotion_dicts[emotion][str(row[col_name]).lower()] = str(correct)
            
            # Add correct spelling
            emotion_dicts[emotion][str(correct).lower()] = str(correct)
        
        combined_dict = {}
        for d in emotion_dicts.values():
            combined_dict.update(d)
        
        return combined_dict
    
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return {}

# 2️⃣ Normalize a single text
def normalize_text(text, spelling_dict):
    if not text or pd.isna(text):
        return text
    
    words = str(text).split()
    normalized_words = [spelling_dict.get(word.lower(), word) for word in words]
    return ' '.join(normalized_words)

# 3️⃣ Load spelling dictionary
csv_path = '/kaggle/input/input-filess/emotion_spelling_final.txt'
spelling_dict = load_emotion_dictionaries(csv_path)
print(f"Loaded {len(spelling_dict)} spelling mappings")

# 4️⃣ Normalize training data (text file)
training_input = '/kaggle/input/input-filess/training_data_sarcasam.txt'
training_output = 'normalized_training_data.txt'

try:
    with open(training_input, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    normalized_training = [normalize_text(line.strip(), spelling_dict) for line in lines]
    
    with open(training_output, 'w', encoding='utf-8') as f:
        for line in normalized_training:
            f.write(line + '\n')
    
    print(f"Normalized training data saved: {training_output} ({len(normalized_training)} lines)")
except Exception as e:
    print(f"Error processing training data: {e}")

# 5️⃣ Normalize testing data (text file - NOT CSV)
testing_input = '/kaggle/input/input-filess/testing_data_sarcasam.txt'
testing_output = 'normalized_testing_data.txt'

try:
    with open(testing_input, 'r', encoding='utf-8') as f:
        test_lines = f.readlines()
    
    normalized_testing = [normalize_text(line.strip(), spelling_dict) for line in test_lines]
    
    with open(testing_output, 'w', encoding='utf-8') as f:
        for line in normalized_testing:
            f.write(line + '\n')
    
    print(f"Normalized testing data saved: {testing_output} ({len(normalized_testing)} lines)")
except Exception as e:
    print(f"Error processing testing data: {e}")

print("✅ Normalization complete!")

# 6️⃣ Display file paths for download
print("\n📁 Final normalized files created:")
print(f"✅ Training: {training_output}")
print(f"✅ Testing: {testing_output}")

# For Kaggle - files are in current directory
# You can download them from the "Output" section in Kaggle
print("\n📥 Files are ready in the output directory!")
print("Go to 'Output' tab → click the download icon next to each file")

# Optional: Also create a summary
summary_file = 'normalization_summary.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("NORMALIZATION SUMMARY\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Spelling Dictionary Loaded: {len(spelling_dict)} mappings\n")
    f.write(f"Training Lines Processed: {len(normalized_training)}\n")
    f.write(f"Testing Lines Processed: {len(normalized_testing)}\n\n")
    f.write(f"Output Files:\n")
    f.write(f"1. {training_output}\n")
    f.write(f"2. {testing_output}\n")

print(f"\n📄 Summary file created: {summary_file}")

Loaded 1804 spelling mappings
Normalized training data saved: normalized_training_data.txt (1824 lines)
Normalized testing data saved: normalized_testing_data.txt (1956 lines)
✅ Normalization complete!

📁 Final normalized files created:
✅ Training: normalized_training_data.txt
✅ Testing: normalized_testing_data.txt

📥 Files are ready in the output directory!
Go to 'Output' tab → click the download icon next to each file

📄 Summary file created: normalization_summary.txt
